# Phase 7 — MMDetection: does enhancement improve detection in fog and at night?

This notebook produces the main thesis table with **MMDetection 3.3**.

* **Detector:** RTMDet-s, COCO-pretrained, from the official OpenMMLab model zoo. It is the same model the deployed Space runs, and it needs no training.
* **Fog:** RTTS, real hazy images. Enhancement options: none, CLAHE, FFA-Net.
* **Night:** ExDark, real low-light images. Enhancement options: none, CLAHE, Zero-DCE++.
* **Metrics:** COCO mAP, mAP@0.5, mAP@0.75, per-class AP, enhancement ms/img, detector ms/img and FPS, parameters, and peak GPU memory.

Only the enhancement changes between runs. The detector and images stay the same, so a change in mAP comes from the enhancement. This answers research question 1.

**Runtime:** select GPU (T4) under *Runtime → Change runtime type*. Results are written to Google Drive after every run. If Colab disconnects, re-run all cells and the finished runs are skipped.

In [ ]:
# 0. Drive + repo
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/computer_vision_expirement'
RESULTS = '/content/drive/MyDrive/computer_vision/results/mmdet_eval'
os.makedirs(RESULTS, exist_ok=True)
# Scripts for this notebook live on this branch until it is merged into main.
# If the branch no longer exists (merged + deleted), main is used instead.
BRANCH = 'feat/mmdet-eval'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/Ib-Programmer/computer_vision_expirement.git {REPO}
%cd {REPO}
!git fetch -q origin && (git checkout -q {BRANCH} 2>/dev/null && git pull -q origin {BRANCH}) || (git checkout -q main && git pull -q origin main)
!git branch --show-current && git log --oneline -1

## 1. MMDetection environment

Colab ships Python 3.12, but MMDetection 3.3 needs mmcv 2.1, whose prebuilt CUDA wheels cover Python ≤ 3.11 and torch 2.1. This cell builds a separate Python 3.10 environment with `uv`, which takes about 2–3 minutes. It is cached, so re-running the cell is instant. Colab's own Python is not touched.

In [ ]:
ENV = '/content/mmenv'
PY = f'{ENV}/bin/python'
MMCV_WHL = ('https://download.openmmlab.com/mmcv/dist/cu121/torch2.1.0/'
            'mmcv-2.1.0-cp310-cp310-manylinux1_x86_64.whl')
if not os.path.exists(PY):
    !pip install -q uv
    !uv venv {ENV} --python 3.10 -q
    !uv pip install -q --python {PY} torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
    !uv pip install -q --python {PY} "numpy<2" "opencv-python<4.11" mmengine==0.10.7 mmdet==3.3.0 {MMCV_WHL} pycocotools pyyaml einops huggingface_hub gdown
!{PY} -c "import torch, mmcv, mmdet, mmcv.ops; print('torch', torch.__version__, '| mmcv', mmcv.__version__, '| mmdet', mmdet.__version__, '| CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

## 2. Enhancement weights and datasets

* **Weights:** Zero-DCE++ and FFA-Net are downloaded from the project's Hugging Face model repo.
* **RTTS:** `fuhao7i/RTTS` on Hugging Face. `RTTS_SPLIT='test'` uses 433 images. Set it to `'all'` for all 4,322; FFA-Net then takes about 10× longer.
* **ExDark:** `dronefreak/ExDark` on Hugging Face, test split, about 730 images.

Each dataset's classes are mapped to their COCO names, so the COCO-pretrained detector is scored directly. For example, *motorbike* becomes *motorcycle* and *People* becomes *person*.

In [ ]:
RTTS_SPLIT = 'test'

from huggingface_hub import hf_hub_download
os.makedirs(f'{REPO}/weights', exist_ok=True)
for f in ['zero_dce_pp.pth', 'ffa_net_outdoor.pk']:
    hf_hub_download('IbProgrammmer/cv-thesis-models', f, local_dir=f'{REPO}/weights')
!ls -la {REPO}/weights

!python scripts/prepare_eval_datasets.py --dataset all --split {RTTS_SPLIT}

## 3. Smoke test (20 images, about 1 minute)

This checks the whole chain before the long runs: environment, weights, datasets, enhancement, MMDetection evaluation and timing. It writes to a separate file, so the real results stay clean.

In [ ]:
SMOKE = f'{RESULTS}/smoke_test.jsonl'
!rm -f {SMOKE}
!{PY} scripts/mmdet_eval.py --dataset rtts --enhance none --limit 20 --out {SMOKE} --no-benchmark
!{PY} scripts/mmdet_eval.py --dataset rtts --enhance ffanet --limit 20 --out {SMOKE}
!{PY} scripts/mmdet_eval.py --dataset exdark --enhance zero_dce --limit 20 --out {SMOKE}
!cat {SMOKE}

## 4. Main experiment: RTMDet-s × enhancement (the thesis table)

On a T4, the none, CLAHE and Zero-DCE++ runs take a few minutes each. FFA-Net is the slow one, at about 1 s per image. Restormer is off by default because its available weights were trained for **deraining**, not fog or night. Set `RUN_RESTORMER = True` to include it as an extra baseline.

In [ ]:
OUT = f'{RESULTS}/mmdet_eval.jsonl'
MODEL = 'rtmdet_s_8xb32-300e_coco'
RUN_RESTORMER = False

MATRIX = {
    'rtts':   ['none', 'clahe', 'ffanet'],
    'exdark': ['none', 'clahe', 'zero_dce'],
}
if RUN_RESTORMER:
    MATRIX['rtts'].append('restormer')

for dataset, methods in MATRIX.items():
    for enh in methods:
        print(f'\n===== {MODEL} | {dataset} | {enh} =====')
        !{PY} scripts/mmdet_eval.py --model {MODEL} --dataset {dataset} --enhance {enh} --out {OUT} --skip-existing

## 5. Optional: detector comparison inside MMDetection

The proposal compares several detectors. This cell runs other COCO-pretrained MMDetection models on the same images, without enhancement and with each dataset's main enhancer. It shows whether the enhancement effect holds across detectors, and it records each model's speed and size for the accuracy-versus-speed trade-off (research question 3). Allow about 15–25 minutes per model.

In [ ]:
RUN_DETECTOR_COMPARISON = False
OTHER_MODELS = [
    'rtmdet_tiny_8xb32-300e_coco',
    'faster-rcnn_r50_fpn_1x_coco',
    'retinanet_r50_fpn_1x_coco',
    'ssd300_coco',
]
MAIN_ENHANCER = {'rtts': 'ffanet', 'exdark': 'zero_dce'}

if RUN_DETECTOR_COMPARISON:
    for m in OTHER_MODELS:
        for dataset, enh in MAIN_ENHANCER.items():
            for e in ['none', enh]:
                print(f'\n===== {m} | {dataset} | {e} =====')
                !{PY} scripts/mmdet_eval.py --model {m} --dataset {dataset} --enhance {e} --out {OUT} --skip-existing

## 6. Results tables

The tables are saved to Drive as CSV and Markdown, ready to paste into the thesis. **Δ mAP** is the change against no enhancement, for the same detector and dataset.

In [ ]:
!pip install -q tabulate
import json
import pandas as pd

rows = [json.loads(l) for l in open(OUT) if l.strip()]
df = pd.DataFrame(rows)
df = df.sort_values('time').drop_duplicates(['model', 'dataset', 'enhance'], keep='last')
df = df.sort_values(['model', 'dataset', 'time'], kind='stable').reset_index(drop=True)

base = df[df.enhance == 'none'].set_index(['model', 'dataset'])['mAP']
df['delta_mAP'] = df.apply(lambda r: r.mAP - base.get((r.model, r.dataset), float('nan')), axis=1)

pct = lambda x: None if pd.isna(x) else round(100 * x, 1)
acc = df[['model', 'dataset', 'enhance', 'n_images', 'mAP', 'mAP50', 'mAP75', 'delta_mAP', 'enhance_ms_per_img']].copy()
for c in ['mAP', 'mAP50', 'mAP75', 'delta_mAP']:
    acc[c] = acc[c].map(pct)
acc = acc.rename(columns={'mAP': 'mAP %', 'mAP50': 'mAP@0.5 %', 'mAP75': 'mAP@0.75 %',
                          'delta_mAP': 'Δ mAP (pts)', 'enhance_ms_per_img': 'enhance ms/img'})
print('Table 1 - Detection accuracy with and without enhancement')
display(acc)

speed_cols = [c for c in ['model', 'dataset', 'det_ms_per_img', 'fps', 'params_M', 'gpu_mem_MB', 'device', 'zoo_coco_box_AP'] if c in df.columns]
speed = df[df.enhance == 'none'][speed_cols]
print('\nTable 2 - Detector cost (no enhancement)')
display(speed)

pc = df.set_index(['model', 'dataset', 'enhance'])['per_class_AP'].apply(pd.Series).mul(100).round(1)
print('\nTable 3 - Per-class AP %')
display(pc)

acc.to_csv(f'{RESULTS}/table1_accuracy.csv', index=False)
speed.to_csv(f'{RESULTS}/table2_speed.csv', index=False)
pc.to_csv(f'{RESULTS}/table3_per_class.csv')
with open(f'{RESULTS}/tables.md', 'w', encoding='utf-8') as f:
    f.write('## Table 1 - Detection accuracy with and without enhancement\n\n' + acc.to_markdown(index=False) + '\n\n')
    f.write('## Table 2 - Detector cost (no enhancement)\n\n' + speed.to_markdown(index=False) + '\n\n')
    f.write('## Table 3 - Per-class AP %\n\n' + pc.to_markdown() + '\n')
print('\nSaved to', RESULTS)

In [ ]:
import matplotlib.pyplot as plt

main = df[df.model == MODEL]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (dataset, title) in zip(axes, [('rtts', 'RTTS (real fog)'), ('exdark', 'ExDark (low light)')]):
    d = main[main.dataset == dataset].set_index('enhance').reindex(MATRIX[dataset]).dropna(subset=['mAP'])
    ax.bar(d.index, d.mAP50 * 100, color='#4C72B0')
    for i, v in enumerate(d.mAP50 * 100):
        ax.text(i, v + 0.5, f'{v:.1f}', ha='center')
    ax.set_ylim(0, max(d.mAP50.max() * 100 * 1.15, 1))
    ax.set_title(title)
    ax.set_xlabel('enhancement')
axes[0].set_ylabel('mAP@0.5 (%)')
fig.suptitle(f'{MODEL} (MMDetection) - effect of enhancement')
fig.tight_layout()
fig.savefig(f'{RESULTS}/fig_enhancement_map50.png', dpi=200)
plt.show()

## 7. Notes for the thesis

* **Framework:** all detection results come from MMDetection 3.3: the model zoo, `Runner.test()` and `CocoMetric`. The deployed Space runs the same RTMDet-s through `mmdet.apis`.
* **No fine-tuning:** the detector is COCO-pretrained. RTTS and ExDark are used only for testing, so the enhancement comparison is a fair "plug-in preprocessing" test.
* **Class mapping:** dataset classes are mapped to their COCO names. A vehicle labelled *car* in RTTS but predicted as *truck* counts as a miss. This affects every method equally, so it does not bias the comparison.
* **ExDark copy:** the Hugging Face mirror is a Roboflow export with images resized to 640×640. Its labels are the original annotations.
* **Speed:** the speed numbers are measured on the Colab GPU. The Space runs on CPU, so its per-request latency is higher.